In [1]:
import networkx as nx
from shapely.geometry import Point, Polygon, box
from shapely import intersection, intersection_all, difference, set_precision
from jax.random import ball, key, split
import random
import math
import bisect
import heapq
import time
import importlib
import numpy as np
from collections import Counter

In [2]:
!pip install igraph

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
import igraph as ig

#ICH

In [9]:
import sys
sys.path.append('content')
import multilateration
importlib.reload(multilateration)
from multilateration import ich

# Shared functions

In [10]:
def euclidean_distance(p, q):
    return math.sqrt((p[0]-q[0])**2 + (p[1]-q[1])**2)

In [11]:
def checkIfResolvingSet_igraph(g, S, dist_matrix):
    if not S:
        return False
    nodes = list(range(g.vcount()))
    representations = {}
    for v in nodes:
        rep = tuple(dist_matrix[s][v] for s in S)
        if rep in representations:
            return False
        representations[rep] = v
    return True

In [12]:
def entropy_of_landmark_candidate(nodes, candidate, dist_matrix):
    cnt = Counter()
    for v in nodes:
        d = dist_matrix[v][candidate]
        code = int(d) if d < float('inf') else -1
        cnt[code] += 1
    total = sum(cnt.values())
    H = 0.0
    for c in cnt.values():
        p = c / total
        H -= p * math.log2(p)
    return H

In [13]:
def build_zobrist_tables(landmarks, dist_matrix, INF_CODE, bits=64):
    max_dist = INF_CODE
    rand = [
        [random.getrandbits(bits) for _ in range(max_dist+1)]
        for _ in range(len(landmarks))
    ]
    return rand

In [14]:
def compute_initial_hashes(n, landmarks, dist_matrix, rand, INF_CODE):
    hashes = [0] * n
    for v in range(n):
        h = 0
        for j, l in enumerate(landmarks):
            d = dist_matrix[v][l]
            code = int(d) if d < float('inf') else INF_CODE
            h ^= rand[j][code]
        hashes[v] = h
    return hashes

In [15]:
def try_remove_landmark(j, l, n, hashes, rand, dist_matrix, INF_CODE):
    adjusted = [0] * n
    for v in range(n):
        d = dist_matrix[v][l]
        code = int(d) if d < float('inf') else INF_CODE
        adjusted[v] = hashes[v] ^ rand[j][code]
    if len(set(adjusted)) == n:
        return adjusted
    return None

In [16]:
def prune_resolving_set_zobrist_fast(g, resolving_set, dist_matrix, bits=64):
    n = g.vcount()
    landmarks = list(resolving_set)
    if not landmarks:
        return set()
    max_d = 0
    for v in range(n):
        for l in landmarks:
            d = dist_matrix[v][l]
            if d < float('inf'):
                max_d = max(max_d, int(d))
    INF_CODE = max_d + 1

    rand = build_zobrist_tables(landmarks, dist_matrix, INF_CODE, bits=bits)
    hashes = compute_initial_hashes(n, landmarks, dist_matrix, rand, INF_CODE)

    pruned = set(landmarks)
    changed = True

    while changed:
        changed = False
        for j, l in enumerate(landmarks):
            if l not in pruned:
                continue

            adjusted = try_remove_landmark(j, l, n, hashes, rand, dist_matrix, INF_CODE)
            if adjusted is not None:
                pruned.remove(l)
                hashes = adjusted
                changed = True
                break

    return pruned

# New Unit Square Class

In [17]:
class UnitSquareNew:
    """
    Heuristic point generator for the grid-based metric dimension algorithm.

    The two fitted curves (defined in the canonical frame centered at (cx, cy))
    are:
        curve1: y = -0.028964*(x-cx)^2 + 0.985804*(x-cx) + 0.630070 + cy
        curve2: y =  0.021721*(x-cx)^2 + 0.981117*(x-cx) - 0.626609 + cy

    point_generator yields ideal probe points in the order dictated by the
    pattern in pattern.pdf:
      1. Center of curve1 (x=cx), then center of curve2 (x=cx).
      2. Expand symmetrically left/right in steps of d, yielding
         (x+d, curve1), (x+d, curve2), (x-d, curve1), (x-d, curve2), ...
      A point is only yielded if it falls inside the grid cell's bounding
      square [cx-r, cx+r] x [cy-r, cy+r].  The generator stops once a full
      expansion step produces no in-bounds points on either curve in either
      direction.

    Yields: (x, y) tuples (curve_id dropped; callers only need coordinates).
    """

    @staticmethod
    def _curve1_y(x, cx, cy):
        dx = x - cx
        return -0.028964 * dx**2 + 0.985804 * dx + 0.630070 + cy

    @staticmethod
    def _curve2_y(x, cx, cy):
        dx = x - cx
        return  0.021721 * dx**2 + 0.981117 * dx - 0.626609 + cy

    @staticmethod
    def _in_square(x, y, cx, cy, r):
        """True iff (x, y) is inside the axis-aligned cell square of half-side r."""
        return (cx - r) <= x <= (cx + r) and (cy - r) <= y <= (cy + r)

    def point_generator(self, cx, cy, r, d):
        """
        Generator of ideal probe points for the cell centred at (cx, cy).

        Parameters
        ----------
        cx, cy : float  -- cell centre in RGG [0,1]^2 space
        r      : float  -- RGG radius (half-side of the grid cell)
        d      : float  -- step size along x between successive probe points
        """
        # --- Step 0: centre points (x = cx) ---
        y1_center = self._curve1_y(cx, cx, cy)
        if self._in_square(cx, y1_center, cx, cy, r):
            yield (cx, y1_center)

        y2_center = self._curve2_y(cx, cx, cy)
        if self._in_square(cx, y2_center, cx, cy, r):
            yield (cx, y2_center)

        # --- Steps 1, 2, 3, ...: expand symmetrically ---
        step = 1
        while True:
            any_yielded = False

            for sign in [1, -1]:
                x = cx + sign * step * d

                y1 = self._curve1_y(x, cx, cy)
                if self._in_square(x, y1, cx, cy, r):
                    yield (x, y1)
                    any_yielded = True

                y2 = self._curve2_y(x, cx, cy)
                if self._in_square(x, y2, cx, cy, r):
                    yield (x, y2)
                    any_yielded = True

            if not any_yielded:
                return

            step += 1


# Grid method

In [18]:
def generate_rgg_with_grid(G, m, n):
    grid = {(r, c): [] for r in range(m) for c in range(n)}
    pos = {}

    for node in range(G.vcount()):
        coord = G.vs[node]["pos"]
        x, y = coord[0], coord[1]

        pos[node] = (x, y)

        col = min(int(x * n), n - 1)
        row = min(int(y * m), m - 1)
        grid[(row, col)].append(node)

    return grid

In [19]:
def cell_center(i, j, m, n):
    """
    Returns the (x, y) center coordinate of the cell at row i, column j.

    Grid spans [0,1] x [0,1], divided into m rows and n columns.
    """
    x = (j + 0.5) / n
    y = (i + 0.5) / m
    return x, y

In [20]:
def get_nodes_in_cell(grid: dict, row: int, col: int) -> list:
    return grid.get((row, col), [])

In [21]:
def get_metric_dimension_of_graph_with_pruning_igraph_grid(G, r, k_nearest=1):
    g = ig.Graph.from_networkx(G)
    nodes_set = set(range(g.vcount()))
    dist_matrix = g.distances()
    resolving_set = set()
    resolved_nodes = set()
    m = n = math.ceil(1/(2*r))
    d = 1 / max(m, n)
    grid = generate_rgg_with_grid(g,m,n)
    for i in range(m):
      if checkIfResolvingSet_igraph(g, resolving_set, dist_matrix) or not nodes_set:
            break
      for j in range(n):
        if checkIfResolvingSet_igraph(g, resolving_set, dist_matrix) or not nodes_set:
            break
        temp__resolving_set,_,nodes_set = get_metric_dimension_of_unit_square(g,grid,i,j,m,n,r,d,dist_matrix,nodes_set,k_nearest)
        print("Number of nodes added for grid is: "+str(len(temp__resolving_set)))
        resolving_set.update(temp__resolving_set)
        newly_added = list(temp__resolving_set)
        #resolving_set = incremental_global_prune(
         #   g, resolving_set, newly_added, dist_matrix
        #)
    resolving_set = prune_resolving_set_zobrist_fast(g, resolving_set, dist_matrix)
    return resolving_set

In [22]:
def get_metric_dimension_of_unit_square(g, grid, i, j, m, n, r, d, dist_matrix, nodes_set, k_nearest):
    """
    Build a local resolving set for the nodes inside grid cell (i, j).

    The UnitSquareNew generator drives landmark selection: each ideal probe
    point narrows the search to the k_nearest real nodes, from which the
    highest-entropy candidate is chosen.  When the generator is exhausted
    before all nodes are resolved we fall back to a pure entropy-greedy pick
    so the loop always terminates cleanly.  A local Zobrist prune removes
    any redundant landmarks before returning.
    """
    pos = g.vs["pos"]
    nodes_within = get_nodes_in_cell(grid, i, j)
    if not nodes_within:
        return {}, 0, nodes_set

    nodes_set = nodes_set - set(nodes_within)
    resolving_set = set()

    cx, cy = cell_center(i, j, m, n)
    us = UnitSquareNew()
    gen = us.point_generator(cx, cy, r, d)

    nodes_remaining = set(nodes_within)

    while nodes_remaining:
        ideal_point = next(gen, None)

        if ideal_point is not None:
            # Heuristic-guided: find the k nearest nodes to the ideal point,
            # then pick the one with the highest entropy among remaining nodes.
            candidates = sorted(
                nodes_remaining,
                key=lambda v: euclidean_distance(pos[v], ideal_point)
            )[:min(len(nodes_remaining), k_nearest)]
            best = max(
                candidates,
                key=lambda c: entropy_of_landmark_candidate(nodes_remaining, c, dist_matrix)
            )
        else:
            # Generator exhausted before all nodes resolved: fall back to
            # pure entropy-greedy over all remaining nodes.
            best = max(
                nodes_remaining,
                key=lambda c: entropy_of_landmark_candidate(nodes_remaining, c, dist_matrix)
            )

        resolving_set.add(best)
        nodes_remaining.remove(best)

        # Early-exit: check if remaining nodes are already distinguished.
        if nodes_remaining:
            signatures = {
                v: tuple(dist_matrix[v][l] for l in resolving_set)
                for v in nodes_remaining
            }
            if len(signatures) == len(set(signatures.values())):
                break
    return resolving_set, len(resolving_set),nodes_set
    #pruned_local = prune_local_zobrist(list(nodes_within), resolving_set, dist_matrix)
    #return pruned_local, len(pruned_local), nodes_set


# Original Zoomed In Circle Method

In [23]:
class UnitSquare:
  def __init__(self):
    self.vertical_cuts_x = [0.0,1.0]
    self.horizontal_cuts_y = [0.0,1.0]
    self.divisions = {}
    self.priority_queue = []
    self.add_new_division(0,0)

  def add_new_division(self,i,j):
    x0, x1 = self.vertical_cuts_x[i], self.vertical_cuts_x[i+1]
    y0, y1 = self.horizontal_cuts_y[j], self.horizontal_cuts_y[j+1]
    area = (x1 - x0) * (y1 - y0)
    self.divisions[(i,j)] = (area,(x0,x1,y0,y1))
    heapq.heappush(self.priority_queue,(-area,i,j))

  def pop_largest_division(self):
    while self.priority_queue:
      area,i,j = heapq.heappop(self.priority_queue)
      if (i,j) in self.divisions and -area == self.divisions[(i,j)][0]:
        return i,j
    return None

  def get_cut_direction(self,x0,x1,y0,y1):
    width = x1 - x0
    height = y1 - y0
    return 'vertical' if width>height else 'horizontal'

  def add(self):
    popped_division = self.pop_largest_division()
    if popped_division is None:
      return
    i,j = popped_division
    (x0,x1,y0,y1) = self.divisions[(i,j)][1]
    del self.divisions[(i,j)]
    direction = self.get_cut_direction(x0,x1,y0,y1)
    if direction=='vertical':
      cut_x = (x0+x1)/2
      bisect.insort(self.vertical_cuts_x,cut_x)
      new_x = self.vertical_cuts_x.index(cut_x)
      for temp_x in range(new_x-1,len(self.vertical_cuts_x)-1):
        for temp_y in range(len(self.horizontal_cuts_y)-1):
          if (temp_x,temp_y) in self.divisions:
              del self.divisions[(temp_x,temp_y)]
          self.add_new_division(temp_x,temp_y)
      return cut_x,False
    else:
      cut_y = (y0+y1)/2
      bisect.insort(self.horizontal_cuts_y,cut_y)
      new_y = self.horizontal_cuts_y.index(cut_y)
      for temp_y in range(new_y-1,len(self.horizontal_cuts_y)-1):
        for temp_x in range(len(self.vertical_cuts_x)-1):
          if (temp_x,temp_y) in self.divisions:
              del self.divisions[(temp_x,temp_y)]
          self.add_new_division(temp_x,temp_y)
      return cut_y, True


  def get_all_areas(self):
    areas = []
    for area_and_coordinates in self.divisions.values():
      area, (x0, x1, y0, y1) = area_and_coordinates
      areas.append(area)
    return areas

  def get_probability(self):
    result = 0
    areas = self.get_all_areas()
    for area in areas:
      result = result + (area**2)
    return 1-result


class UnitCircle:
    def __init__(self,center_x,center_y,r):
        self.us = UnitSquare()
        self.centers = []
        self.center_x=center_x
        self.center_y = center_y
        self.r = r
    def add(self):
        coordinate, is_horizontal = self.us.add()
        return self.modifyCenters(coordinate, is_horizontal)

    def modifyCenters(self, coordinate, is_horizontal):
        if not is_horizontal:
            new_x = 0
            if(coordinate>0.5):
              new_x = self.center_x + ((1-coordinate)*2*self.r)
            else:
              new_x = self.center_x + (coordinate*2*self.r)
            self.centers.append((new_x , self.center_y))
            return (new_x, self.center_y)
        else:
            new_y = 0
            if(coordinate>0.5):
                new_y = self.center_y + ((1-coordinate)*2*self.r)
            else:
                new_y = self.center_y + (coordinate*2*self.r)
            self.centers.append((self.center_x, new_y))
            return (self.center_x, new_y)
    def getProbability(self):
        base_circle = Point(self.center_x,self.center_y).buffer(1)
        cut_circles = [Point(x,y).buffer(1) for (x,y) in self.centers]
        regions = [base_circle]
        for cut_circle in cut_circles:
          new_regions = []
          for region in regions:
            inter = region.intersection(cut_circle)
            if not inter.is_empty:
              new_regions.append(inter)
            diff = region.difference(cut_circle)
            if not diff.is_empty:
              new_regions.append(diff)
          regions = new_regions
        total_area = base_circle.area
        area_fractions = [region.area/total_area for region in regions]
        p_same_region = sum(area**2 for area in area_fractions)
        return 1-p_same_region
    def getCenters(self):
      return self.centers

In [24]:
def get_metric_dimension_of_graph_with_pruning_igraph_circle(G, r, k_nearest=1, max_iters=1000):
    g = ig.Graph.from_networkx(G)
    nodes_set = set(range(g.vcount()))
    dist_matrix = g.distances()
    resolving_set = set()
    iter_count = 0
    while iter_count < max_iters:
        if checkIfResolvingSet_igraph(g, resolving_set, dist_matrix):
            break
        if not nodes_set:
            break

        degrees = [g.degree(n) for n in nodes_set]
        temp_center = (
            random.choices(list(nodes_set), weights=degrees, k=1)[0]
            if sum(degrees) > 0 else random.choice(list(nodes_set))
        )

        temp_set, _, nodes_set = get_metric_dimension_of_unit_circle_igraph_new(
            g, r, temp_center, dist_matrix, nodes_set,k_nearest)

        resolving_set.update(temp_set)

        newly_added = list(temp_set)
       # resolving_set = incremental_global_prune(
        #    g, resolving_set, newly_added, dist_matrix
        #)

        iter_count += 1

    resolving_set = prune_resolving_set_zobrist_fast(g, resolving_set, dist_matrix)
    return resolving_set

In [25]:
def get_metric_dimension_of_unit_circle_igraph_new(g, r, temp_center, dist_matrix, node_set, k_nearest):
    pos = g.vs["pos"]
    nodes_within = [i for i, d in enumerate(dist_matrix[temp_center]) if i in node_set and d == 1]
    if not nodes_within:
        node_set.remove(temp_center)
        return {temp_center}, 1, node_set
    if temp_center not in nodes_within:
        nodes_within.append(temp_center)
    node_set = node_set - set(nodes_within)
    resolving_set = set()
    x_center, y_center = pos[temp_center]
    uc = UnitCircle(x_center, y_center, r)
    nodes_remaining = set(nodes_within)
    while nodes_remaining:
        ideal_point = uc.add()
        candidates = sorted(
            nodes_remaining,
            key=lambda v: euclidean_distance(pos[v], ideal_point)
        )[:min(len(nodes_remaining),k_nearest)]
        best = max(candidates, key=lambda c: entropy_of_landmark_candidate(nodes_remaining, c, dist_matrix))
        resolving_set.add(best)
        nodes_remaining.remove(best)
        signatures = {}
        for v in nodes_remaining:
            sig = tuple(dist_matrix[v][l] for l in resolving_set)
            signatures[v] = sig
        if len(signatures) == len(set(signatures.values())):
            break
    return resolving_set, len(resolving_set), node_set
    #pruned_local = prune_local_zobrist(list(nodes_within), resolving_set, dist_matrix)

    #return pruned_local, len(pruned_local), node_set

# Zoomed In Square Method

In [26]:
def get_metric_dimension_of_graph_with_pruning_igraph_zoomed_square(G, r, k_nearest=1, max_iters=1000):
    g = ig.Graph.from_networkx(G)
    nodes_set = set(range(g.vcount()))
    dist_matrix = g.distances()
    resolving_set = set()
    iter_count = 0

    while iter_count < max_iters:
        if checkIfResolvingSet_igraph(g, resolving_set, dist_matrix):
            break
        if not nodes_set:
            break

        degrees = [g.degree(n) for n in nodes_set]
        temp_center = (
            random.choices(list(nodes_set), weights=degrees, k=1)[0]
            if sum(degrees) > 0 else random.choice(list(nodes_set))
        )

        temp_set, _, nodes_set = get_metric_dimension_of_unit_square_zoomed_square(
            g, r, temp_center, dist_matrix, nodes_set, k_nearest)

        resolving_set.update(temp_set)
        iter_count += 1

    resolving_set = prune_resolving_set_zobrist_fast(g, resolving_set, dist_matrix)
    return resolving_set

In [27]:
def get_metric_dimension_of_unit_square_zoomed_square(g, r, temp_center, dist_matrix, nodes_set, k_nearest):
    """
    Build a local resolving set for the nodes within graph distance 1 of
    temp_center, using the UnitSquareNew heuristic to guide probe point
    selection. Falls back to entropy-greedy when the generator is exhausted.
    """
    pos = g.vs["pos"]
    nodes_within = [i for i, d in enumerate(dist_matrix[temp_center]) if i in nodes_set and d == 1]
    if not nodes_within:
        nodes_set.remove(temp_center)
        return {temp_center}, 1, nodes_set
    if temp_center not in nodes_within:
        nodes_within.append(temp_center)
    nodes_set = nodes_set - set(nodes_within)
    resolving_set = set()

    x_center, y_center = pos[temp_center]
    us = UnitSquareNew()
    d = 2 * r
    gen = us.point_generator(x_center, y_center, r, d)

    nodes_remaining = set(nodes_within)

    while nodes_remaining:
        ideal_point = next(gen, None)

        if ideal_point is not None:
            candidates = sorted(
                nodes_remaining,
                key=lambda v: euclidean_distance(pos[v], ideal_point)
            )[:min(len(nodes_remaining), k_nearest)]
            best = max(
                candidates,
                key=lambda c: entropy_of_landmark_candidate(nodes_remaining, c, dist_matrix)
            )
        else:
            best = max(
                nodes_remaining,
                key=lambda c: entropy_of_landmark_candidate(nodes_remaining, c, dist_matrix)
            )

        resolving_set.add(best)
        nodes_remaining.remove(best)

        if nodes_remaining:
            signatures = {
                v: tuple(dist_matrix[v][l] for l in resolving_set)
                for v in nodes_remaining
            }
            if len(signatures) == len(set(signatures.values())):
                break

    return resolving_set, len(resolving_set), nodes_set

# Test

In [28]:
n=10
r=0.1

In [29]:
G = nx.random_geometric_graph(n, r)

In [30]:
igraph_rs_grid = get_metric_dimension_of_graph_with_pruning_igraph_grid(G,r)

Number of nodes added for grid is: 0
Number of nodes added for grid is: 0
Number of nodes added for grid is: 1
Number of nodes added for grid is: 0
Number of nodes added for grid is: 1
Number of nodes added for grid is: 0
Number of nodes added for grid is: 0
Number of nodes added for grid is: 1
Number of nodes added for grid is: 0
Number of nodes added for grid is: 0
Number of nodes added for grid is: 0
Number of nodes added for grid is: 1
Number of nodes added for grid is: 0
Number of nodes added for grid is: 0
Number of nodes added for grid is: 1
Number of nodes added for grid is: 1
Number of nodes added for grid is: 0
Number of nodes added for grid is: 1
Number of nodes added for grid is: 0
Number of nodes added for grid is: 1
Number of nodes added for grid is: 0
Number of nodes added for grid is: 0
Number of nodes added for grid is: 1


In [31]:
igraph_rs_circle = get_metric_dimension_of_graph_with_pruning_igraph_circle(G,r)

In [32]:
igraph_rs_square = get_metric_dimension_of_graph_with_pruning_igraph_zoomed_square(G,r)

In [33]:
print(len(igraph_rs_grid))

9


In [34]:
print(len(igraph_rs_circle))

9


In [35]:
print(len(igraph_rs_square))

9


In [36]:
ich_rs = ich(G)

In [37]:
print(len(ich_rs))

9


In [38]:
g = ig.Graph.from_networkx(G)
dist_matrix = g.distances()

In [39]:
checkIfResolvingSet_igraph(g,igraph_rs_grid,dist_matrix)

True

In [40]:
checkIfResolvingSet_igraph(g,igraph_rs_circle,dist_matrix)

True

In [41]:
checkIfResolvingSet_igraph(g,igraph_rs_square,dist_matrix)

True

# Benchmark: Grid vs Circle vs Zoomed Square vs ICH

In [42]:
import time
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment

n_values = [10, 50, 100, 200, 500, 1000]
r_values = [0.1, 0.3, 0.5, 0.7, 0.9]

results = []

for n_val in n_values:
    for r_val in r_values:
        print(f"Running n={n_val}, r={r_val}...")
        G = nx.random_geometric_graph(n_val, r_val)
        g = ig.Graph.from_networkx(G)
        dist_matrix = g.distances()

        # Grid method
        t0 = time.time()
        rs_grid = get_metric_dimension_of_graph_with_pruning_igraph_grid(G, r_val)
        t_grid = round(time.time() - t0, 4)
        valid_grid = checkIfResolvingSet_igraph(g, rs_grid, dist_matrix)

        # Circle method
        t0 = time.time()
        rs_circle = get_metric_dimension_of_graph_with_pruning_igraph_circle(G, r_val)
        t_circle = round(time.time() - t0, 4)
        valid_circle = checkIfResolvingSet_igraph(g, rs_circle, dist_matrix)

        # Zoomed Square method
        t0 = time.time()
        rs_square = get_metric_dimension_of_graph_with_pruning_igraph_zoomed_square(G, r_val)
        t_square = round(time.time() - t0, 4)
        valid_square = checkIfResolvingSet_igraph(g, rs_square, dist_matrix)

        # ICH
        t0 = time.time()
        rs_ich = ich(G)
        t_ich = round(time.time() - t0, 4)
        g_ich = ig.Graph.from_networkx(G)
        dm_ich = g_ich.distances()
        valid_ich = checkIfResolvingSet_igraph(g_ich, rs_ich, dm_ich)

        results.append({
            'n': n_val,
            'r': r_val,
            'grid_size': len(rs_grid),
            'grid_time': t_grid,
            'grid_valid': valid_grid,
            'circle_size': len(rs_circle),
            'circle_time': t_circle,
            'circle_valid': valid_circle,
            'square_size': len(rs_square),
            'square_time': t_square,
            'square_valid': valid_square,
            'ich_size': len(rs_ich),
            'ich_time': t_ich,
            'ich_valid': valid_ich,
        })
        print(f"  Grid={len(rs_grid)}({valid_grid}) Circle={len(rs_circle)}({valid_circle}) Square={len(rs_square)}({valid_square}) ICH={len(rs_ich)}({valid_ich})")

print("Done.")

Running n=10, r=0.1...
Number of nodes added for grid is: 1
Number of nodes added for grid is: 1
Number of nodes added for grid is: 0
Number of nodes added for grid is: 0
Number of nodes added for grid is: 0
Number of nodes added for grid is: 0
Number of nodes added for grid is: 0
Number of nodes added for grid is: 1
Number of nodes added for grid is: 0
Number of nodes added for grid is: 1
Number of nodes added for grid is: 0
Number of nodes added for grid is: 0
Number of nodes added for grid is: 0
Number of nodes added for grid is: 0
Number of nodes added for grid is: 1
Number of nodes added for grid is: 0
Number of nodes added for grid is: 0
Number of nodes added for grid is: 1
Number of nodes added for grid is: 0
Number of nodes added for grid is: 1
  Grid=7(True) Circle=6(True) Square=7(True) ICH=6(True)
Running n=10, r=0.3...
Number of nodes added for grid is: 1
Number of nodes added for grid is: 1
Number of nodes added for grid is: 1
Number of nodes added for grid is: 3
  Grid=5(

In [44]:
# Write results to Excel
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Results"

headers = [
    "n",
    "r",
    "Grid Method Result",
    "Grid Method Time (s)",
    "Is Grid Result a Resolving Set",
    "Circle Method Result",
    "Circle Method Time (s)",
    "Is Circle Result a Resolving Set",
    "Zoomed Square Method Result",
    "Zoomed Square Method Time (s)",
    "Is Zoomed Square a Resolving Set",
    "ICH Result",
    "ICH Time (s)",
    "Is ICH Result a Resolving Set",
]

# Header styling
header_font = Font(name='Arial', bold=True, color='FFFFFF')
header_fill = PatternFill('solid', start_color='2F5496')
header_align = Alignment(horizontal='center', vertical='center', wrap_text=True)

for col_idx, h in enumerate(headers, start=1):
    cell = ws.cell(row=1, column=col_idx, value=h)
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = header_align

ws.row_dimensions[1].height = 32

# Data rows
true_fill  = PatternFill('solid', start_color='C6EFCE')  # green for True
false_fill = PatternFill('solid', start_color='FFC7CE')  # red for False
true_font  = Font(name='Arial', color='276221')
false_font = Font(name='Arial', color='9C0006')
data_align = Alignment(horizontal='center')

bool_cols = {5, 8, 11, 14}  # 1-indexed columns that hold True/False

for row_idx, r in enumerate(results, start=2):
    row_data = [
        r['n'], r['r'],
        r['grid_size'],   r['grid_time'],   r['grid_valid'],
        r['circle_size'], r['circle_time'], r['circle_valid'],
        r['square_size'], r['square_time'], r['square_valid'],
        r['ich_size'],    r['ich_time'],    r['ich_valid'],
    ]
    for col_idx, val in enumerate(row_data, start=1):
        cell = ws.cell(row=row_idx, column=col_idx, value=val)
        cell.alignment = data_align
        cell.font = Font(name='Arial')
        if col_idx in bool_cols:
            if val is True:
                cell.fill = true_fill
                cell.font = true_font
            else:
                cell.fill = false_fill
                cell.font = false_font

# Column widths
col_widths = [8, 6, 22, 22, 30, 22, 22, 30, 28, 28, 30, 14, 14, 28]
for col_idx, width in enumerate(col_widths, start=1):
    ws.column_dimensions[openpyxl.utils.get_column_letter(col_idx)].width = width

# Freeze header row
ws.freeze_panes = 'A2'

output_path = 'content/metric_dimension_results.xlsx'
wb.save(output_path)
print(f"Saved to {output_path}")

Saved to content/metric_dimension_results.xlsx
